# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the FAIR² Mental Health Survey dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, in accordance with the Croissant metadata standard.

### Dataset Source
The dataset is sourced via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Set the Croissant metadata URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"\033[1m{metadata.name}\033[0m")
print(metadata.description)
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets and their fields. All entity references use their `@id` values as per the Croissant schema.

In [ ]:
# Get all record sets in the dataset
record_sets = metadata.record_sets

print("Available Record Sets (@id):")
for record_set in record_sets:
    print(f"- {record_set['@id']} : {record_set.get('name', '')}")

# Show all fields for each record set
for record_set in record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    fields = record_set.get('fields', [])
    if not fields:
        print("  No fields defined.")
    else:
        print("  Fields:")
        for field in fields:
            # Each field can be a dict or an @id string if fully described elsewhere
            if isinstance(field, dict):
                print(f"    - {field['@id']}: {field.get('name', '')} ({field.get('dataType', '')})")
            else:
                print(f"    - {field}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

**Note:** We use the record set and field `@id` values found in the previous section.

In [ ]:
# List the record set IDs
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"  Could not load records for {record_set_id}: {e}")

print(f"Loaded dataframes: {list(dataframes.keys())}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps.

We'll use the main record set (typically containing survey responses) for exploration. Please **adjust the `record_set_id` and field `@id`s** below based on record set printed in the previous steps. For this dataset, we expect at least one primary record set for mental health survey responses.

In [ ]:
# Example: Use the first record set for EDA (modify as needed)
if len(dataframes):
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f"Working with primary record set: {main_record_set_id}")
    
    # List numeric fields for possible analysis
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"Numeric fields: {numeric_fields}")
    
    # If GAD-7 or PHQ-9 or similar scoring column found, use it. Else use generic numeric field.
    score_fields = [col for col in df.columns if any(x in col.lower() for x in ['gad7', 'phq9', 'psq', 'score'])]
    if score_fields:
        numeric_field_id = score_fields[0]
    elif numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = df.columns[0]  # fallback
    
    print(f"Using field '{numeric_field_id}' as example numeric field.")

    # Filtering records
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else df.copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        filtered_df[numeric_field_id + '_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Group by key demographics
    demog_fields = [x for x in ['age', 'gender', 'level_of_education', 'marital_status', 'village'] if x in df.columns]
    if len(demog_fields):
        group_field = demog_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"Grouped by {group_field} (mean {numeric_field_id}):")
        display(grouped_df.head())
    else:
        print("No demographic grouping fields found.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Below are example plots for a numeric score field and age if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes):
    # Histogram of the selected numeric field
    plt.figure(figsize=(7, 4))
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.show()
    
    # Boxplot by group if group_field found
    if 'group_field' in locals() and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        plt.figure(figsize=(7, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion

- We have demonstrated the loading and exploration of a machine-actionable Croissant dataset using `mlcroissant`.
- The FAIR² Mental Health Survey dataset contains rich demographic and standardized psychological assessment data from Kilifi County, Kenya.
- Using `mlcroissant` and Python, we loaded records and programmatically explored variable distributions. The methodology can be extended to advanced analyses such as predictive modeling, bias investigation, and FAIRness assessment.
- For more detailed schema inspection, refer to the Croissant JSON at:

  https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json